#### 1차 데이터 전처리

Raw data 전처리에 필요한 함수를 정의

In [1]:
import pandas as pd
import numpy as np
import os
import pickle
import json
import re
import ast
from typing import Dict, List, Tuple, Union
from collections import defaultdict, Counter
from itertools import product

In [2]:
def fix_restart_timestamps(df):
    # 원본 데이터 복사 및 원본 순서 보존을 위한 인덱스 추가
    df = df.copy()
    df['_original_index'] = range(len(df))
    
    result_dfs = []
    
    # SEQID별로 그룹화하여 처리 (원본 순서 유지)
    for _, group in df.groupby('SEQID', sort=False):
        # 원본 순서대로 정렬
        group = group.sort_values('_original_index').reset_index(drop=True)
        # float 타입으로 명시적 변환하여 FutureWarning 방지
        adjusted_timestamps = group['timestamp'].astype(float).copy()
        
        # RESTART 이벤트의 인덱스 찾기
        restart_indices = group[group['event_type'] == 'RESTART'].index.tolist()
        
        for restart_idx in restart_indices:
            # RESTART가 첫 행이 아니고 마지막 행도 아닌 경우에만 처리
            if restart_idx > 0 and restart_idx < len(group) - 1:
                # RESTART 직전 timestamp (이미 조정된 값 사용)
                prev_timestamp = adjusted_timestamps.iloc[restart_idx - 1]
                
                # RESTART 직후 timestamp (원본값 사용)
                next_original_timestamp = group['timestamp'].iloc[restart_idx + 1]
                
                # RESTART의 timestamp 계산 = 직전 + (직후 원본 / 2)
                restart_timestamp = prev_timestamp + (next_original_timestamp / 2).round()
                adjusted_timestamps.iloc[restart_idx] = restart_timestamp
                
                # RESTART 이후부터 END까지 찾기
                end_idx = None
                for i in range(restart_idx + 1, len(group)):
                    if group['event_type'].iloc[i] == 'END':
                        end_idx = i
                        break
                
                # END를 찾지 못하면 그룹의 끝까지 처리
                if end_idx is None:
                    end_idx = len(group) - 1
                
                # RESTART 이후부터 END까지 timestamp 누적
                # offset = RESTART의 조정된 timestamp
                offset = restart_timestamp
                
                for i in range(restart_idx + 1, end_idx + 1):
                    original_timestamp = group['timestamp'].iloc[i]
                    adjusted_timestamps.iloc[i] = offset + original_timestamp
        
        # 조정된 timestamp 적용
        group['timestamp'] = adjusted_timestamps
        result_dfs.append(group)
    
    # 모든 그룹 합치기
    result = pd.concat(result_dfs, ignore_index=True)
    
    # 원본 순서로 정렬 후 임시 인덱스 컬럼 제거
    result = result.sort_values('_original_index').drop('_original_index', axis=1).reset_index(drop=True)
    
    return result

def count_sequences(df, time_window=100):
    """
    SEQID별로 시간 간격 내 이벤트 조합을 카운트
    
    Parameters:
    df: DataFrame (SEQID, event_type, event_description, timestamp)
    time_window: 시간 간격 기준 (default: 100)
    
    Returns:
    Counter: 조합별 카운트
    """
    all_combinations = []
    
    for seqid, group_df in df.groupby('SEQID'):
        group_df = group_df.reset_index(drop=True)
        
        groups = []
        current_group = [0]
        
        for i in range(1, len(group_df)):
            if group_df.loc[i, 'timestamp'] - group_df.loc[current_group[0], 'timestamp'] <= time_window:
                current_group.append(i)
            else:
                groups.append(current_group)
                current_group = [i]
        groups.append(current_group)
        
        # 각 그룹의 조합 추출
        for group in groups:
            combo = tuple(group_df.loc[idx, 'event_type'] for idx in group)
            all_combinations.append(combo)
    
    return Counter(all_combinations)


def merge_patterns(df, pattern_map):
    """
    특정 패턴을 하나의 행으로 병합
    
    Parameters:
    df: DataFrame (SEQID, event_type, event_description, timestamp)
    pattern_map: dict {('A', 'B', 'C'): 'B', ...}  # 값은 유지할 event_type
    
    Returns:
    DataFrame: 병합된 데이터프레임
    """
    result_rows = []
    
    for seqid, group_df in df.groupby('SEQID'):
        group_df = group_df.reset_index(drop=True)
        events = group_df['event_type'].tolist()
        
        i = 0
        while i < len(events):
            matched = False
            
            for pattern, keep_type in sorted(pattern_map.items(), key=lambda x: -len(x[0])):
                pattern_len = len(pattern)
                if i + pattern_len <= len(events):
                    if tuple(events[i:i+pattern_len]) == pattern:
                        # 패턴에서 keep_type 행 찾아서 유지
                        for j in range(pattern_len):
                            if events[i+j] == keep_type:
                                result_rows.append(group_df.iloc[i+j])
                                break
                        i += pattern_len
                        matched = True
                        break
            
            if not matched:
                result_rows.append(group_df.iloc[i])
                i += 1
    
    return pd.DataFrame(result_rows).reset_index(drop=True)

def replace_pattern(df, pattern, new_event_type, new_event_description):
    """
    특정 패턴을 새로운 event_type, event_description으로 변환
    
    Parameters:
    df: DataFrame (SEQID, event_type, event_description, timestamp)
    pattern: tuple ('A', 'B', 'C')
    new_event_type: str 'NEW_TYPE'
    new_event_description: str 'NEW_DESC'
    
    Returns:
    DataFrame: 변환된 데이터프레임
    """
    result_rows = []
    pattern_len = len(pattern)
    
    for seqid, group_df in df.groupby('SEQID'):
        group_df = group_df.reset_index(drop=True)
        events = group_df['event_type'].tolist()
        
        i = 0
        while i < len(events):
            if i + pattern_len <= len(events):
                if tuple(events[i:i+pattern_len]) == pattern:
                    # 첫 번째 행 복사 후 type/description만 변경
                    first_row = group_df.iloc[i].copy()
                    first_row['event_type'] = new_event_type
                    first_row['event_description'] = new_event_description
                    result_rows.append(first_row)
                    i += pattern_len
                    continue
            
            result_rows.append(group_df.iloc[i])
            i += 1
    
    return pd.DataFrame(result_rows).reset_index(drop=True)

def preprocess_keypress_data(df):
    """
    연속된 KEYPRESS 이벤트를 하나로 합치는 전처리 함수
    
    Parameters:
    output_file_path (str): 출력 파일 경로 (.txt)
    """
    
    # SEQID별로 그룹화하여 처리
    processed_groups = []
    
    for seqid in df['SEQID'].unique():
        seqid_data = df[df['SEQID'] == seqid].copy()
        # timestamp 순으로 정렬
        seqid_data = seqid_data.sort_values('timestamp').reset_index(drop=True)
        
        processed_data = process_keypress_sequences(seqid_data)
        processed_groups.append(processed_data)
    
    # 모든 그룹 합치기
    result_df = pd.concat(processed_groups, ignore_index=True)
    
    print(f"처리된 데이터 크기: {len(result_df)} 행")
    print(f"제거된 행 수: {len(df) - len(result_df)}")
    
    return result_df

def process_keypress_sequences(df):
    """
    단일 SEQID 데이터에서 연속된 KEYPRESS 시퀀스 처리
    
    Parameters:
    df (DataFrame): 단일 SEQID의 데이터
    
    Returns:
    DataFrame: 처리된 데이터
    """
    
    if len(df) == 0:
        return df
    
    result_rows = []
    i = 0
    
    while i < len(df):
        current_row = df.iloc[i].copy()
        
        # 현재 행이 KEYPRESS인 경우
        if current_row['event_type'] == 'KEYPRESS':
            keypress_count = 1
            j = i + 1
            
            # 연속된 KEYPRESS 개수 세기
            while j < len(df) and df.iloc[j]['event_type'] == 'KEYPRESS':
                keypress_count += 1
                j += 1
            
            # 첫 번째 KEYPRESS 행의 event_description을 count=N으로 변경
            current_row['event_description'] = f'count={keypress_count}'
            result_rows.append(current_row)
            
            # 다음 처리할 인덱스를 연속된 KEYPRESS 이후로 설정
            i = j
            
        else:
            # KEYPRESS가 아닌 경우 그대로 추가
            result_rows.append(current_row)
            i += 1
    
    return pd.DataFrame(result_rows)

## Raw data 0차 전처리 - 전체 데이터에서 문항별 파일 분리

In [3]:
data = pd.read_csv('US_logdata.txt', sep='\t')

left = range(1, 3)   # 1 ~ 2
right = range(1, 8)  # 1 ~ 7
pairs = product(left, right)

for comb in pairs:
    i, j = comb
    
    mask = (data["booklet_id"] == f"PS{i}") & (data["item_id"] == j)
    prob_data = data.loc[mask].copy()
    
    US_code = prob_data["CNTRYID"].str.split("_", n=1).str[1].str[:2]
    prob_data["SEQID_unify"] = US_code + "_" + prob_data["SEQID"].astype("string")
    
    prob_data.to_csv(f"./input_data/us_ps{i}_{j}.txt", sep='\t', index=False)

## Action event_type, description 변환

1. 각 문항별로 Raw data 전처리
2. 처리된 행동과 각 행동별 빈도 수를 확인

#### PS1_1

In [ ]:
# 사용 예시
problem_num = 'ps1_1'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## 문제 공통 행동 ##
# 최종 제출 프로세스 처리 - BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})

## ps1_1 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'ENVIRONMENT', 'MC_HELP_TOOLBAR', 'MAIL_SENT', 'MAIL_DELETED',
                                    'MC_HELP_MENUITEM', 'SORT_MENU', 'COPY', 'PASTE',
                                    'NEW_FOLDER', 'MC_SORT', 'DELETE_FOLDER'])].copy()

pattern_map = {
    ('FOLDER_VIEWED', 'MAIL_DROP', 'MAIL_MOVED'): 'MAIL_DROP',
    ('FOLDER_VIEWED', 'MAIL_MOVED', 'MAIL_DROP'): 'MAIL_DROP',
    ('BUTTON', 'MAIL_MOVED'): 'BUTTON',
    ('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON',
    ('BREAKOFF', 'END') : 'BREAKOFF'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

처리된 데이터 크기: 30267 행
제거된 행 수: 304


,event_type,timestamp,event_description,SEQID
0,START,0.0,TEST_TIME=328,US_10
1,MAIL_DRAG,32467.0,id=u01a_item101,US_10
2,MAIL_DROP,35606.0,target=u01a_CanComeFolder,US_10
3,FOLDER_VIEWED,36636.0,id=u01a_CanComeFolder,US_10
4,MAIL_DRAG,41961.0,id=u01a_item202,US_10


In [3]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count}회")

MAIL_VIEWED: 7886회
FOLDER_VIEWED: 5994회
MAIL_DRAG: 4109회
BUTTON: 3757회
MAIL_DROP: 2291회
START: 1354회
MAIL_VIEWED → MAIL_DROP: 1351회
TOOLBAR: 710회
MENU: 307회
KEYPRESS: 283회
MENUITEM: 163회
MAIL_VIEWED → FOLDER_VIEWED: 144회
GET_HELP: 115회
TEXTBOX_ONFOCUS: 93회
FOLDER_UNFOLDED: 67회
TEXTBOX_KILLFOCUS: 56회
FOLDER_FOLDED: 54회
KEYPRESS → KEYPRESS: 43회
MAIL_VIEWED → MAIL_VIEWED: 32회
TEXTBOX_KILLFOCUS → BUTTON: 17회
KEYPRESS → MAIL_VIEWED: 16회
TEXTBOX_KILLFOCUS → TEXTBOX_ONFOCUS: 13회
TEXTBOX_KILLFOCUS → TOOLBAR: 12회
RESTART: 6회
TEXTBOX_KILLFOCUS → FOLDER_VIEWED: 6회
KEYPRESS → KEYPRESS → KEYPRESS: 4회
TEXTBOX_ONFOCUS → TEXTBOX_KILLFOCUS: 4회
KEYPRESS → FOLDER_VIEWED: 3회
BREAKOFF: 3회
MAIL_DRAG → MAIL_VIEWED: 1회
KEYPRESS → MAIL_VIEWED → KEYPRESS → MAIL_VIEWED: 1회
BUTTON → BUTTON: 1회
MAIL_DRAG → FOLDER_VIEWED: 1회
RADIO_BTN: 1회
KEYPRESS → MAIL_DRAG: 1회
TEXTBOX_KILLFOCUS → TOOLBAR → TEXTBOX_ONFOCUS: 1회
TEXTBOX_KILLFOCUS → MENU: 1회
TEXTBOX_KILLFOCUS → MENU → TEXTBOX_ONFOCUS: 1회
FOLDER_VIEWED → FOLDER_VIEWE

### PS1_2

In [ ]:
# 사용 예시
problem_num = 'ps1_2'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## 문제 공통 행동 ##
# 최종 제출 프로세스 처리 - BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})

## ps1_2 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'ENVIRONMENT','MC_HELP_TOOLBAR', 'MAIL_SENT', 'MAIL_DELETED',
                                    'MC_HELP_MENUITEM', 'SORT_MENU', 'COPY', 'PASTE', 
                                    'NEW_FOLDER', 'MC_SORT', 'DELETE_FOLDER', 'MAIL_COPIED'])].copy()

pattern_map = {
    ('FOLDER_VIEWED', 'MAIL_DROP', 'MAIL_MOVED'): 'MAIL_DROP',
    ('FOLDER_VIEWED', 'MAIL_MOVED', 'MAIL_DROP'): 'MAIL_DROP',
    ('BUTTON', 'MAIL_MOVED'): 'BUTTON',
    ('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON',
    ('BREAKOFF', 'END') : 'BREAKOFF'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

처리된 데이터 크기: 44523 행
제거된 행 수: 10435


,event_type,timestamp,event_description,SEQID
0,START,0.0,TEST_TIME=8803,US_10
1,FOLDER_VIEWED,53124.0,id=SentFolder,US_10
2,FOLDER_VIEWED,54901.0,id=u01b_InboxFolder,US_10
3,FOLDER_VIEWED,56656.0,id=u01b_CanComeFolder,US_10
4,FOLDER_VIEWED,58119.0,id=u01b_PartyFolder,US_10


In [5]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count}회")

MAIL_VIEWED: 9407회
FOLDER_VIEWED: 8712회
KEYPRESS: 6940회
BUTTON: 4629회
MAIL_DRAG: 4243회
MAIL_DROP: 2464회
KEYPRESS → KEYPRESS: 1890회
MENU: 1698회
MAIL_VIEWED → MAIL_DROP: 1441회
TOOLBAR: 1439회
TEXTBOX_ONFOCUS: 1404회
MENUITEM: 1353회
START: 1351회
TEXTBOX_KILLFOCUS → BUTTON: 665회
TEXTBOX_KILLFOCUS: 401회
FOLDER_UNFOLDED: 311회
FOLDER_FOLDED: 288회
KEYPRESS → KEYPRESS → KEYPRESS: 258회
TEXTBOX_KILLFOCUS → FOLDER_VIEWED: 202회
MAIL_VIEWED → FOLDER_VIEWED: 161회
GET_HELP: 83회
TEXTBOX_KILLFOCUS → TOOLBAR: 67회
TEXTBOX_KILLFOCUS → TEXTBOX_ONFOCUS: 58회
TEXTBOX_KILLFOCUS → MENU: 31회
TEXTBOX_KILLFOCUS → MENUITEM: 24회
KEYPRESS → KEYPRESS → KEYPRESS → KEYPRESS: 16회
KEYPRESS → MAIL_VIEWED: 16회
MAIL_VIEWED → MAIL_VIEWED: 14회
KEYPRESS → FOLDER_VIEWED: 13회
KEYPRESS → KEYPRESS → SHORTCUT: 12회
KEYPRESS → TEXTBOX_KILLFOCUS → TEXTBOX_ONFOCUS: 7회
RESTART: 5회
KEYPRESS → TEXTBOX_KILLFOCUS: 5회
TEXTBOX_KILLFOCUS → TOOLBAR → TEXTBOX_ONFOCUS: 4회
FOLDER_VIEWED → FOLDER_VIEWED: 4회
TEXTBOX_KILLFOCUS → MENU → TEXTBOX_ONFOCUS: 4

### PS1_3

In [ ]:
# 사용 예시
problem_num = 'ps1_3'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## 문제 공통 행동 ##
# 최종 제출 프로세스 처리 - BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})

## ps1_3 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'ENVIRONMENT', 'HISTORY_BACK', 'HISTORY_NEXT', 'SS_SORT','SS_SEARCH', 'BOOKMARK_ADD'])].copy()

pattern_map = {
    ('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON',
    ('BREAKOFF', 'END') : 'BREAKOFF'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

처리된 데이터 크기: 15492 행
제거된 행 수: 625


,event_type,timestamp,event_description,SEQID
0,START,0.0,TEST_TIME=975,US_10
1,TOOLBAR,80057.0,id=webApp,US_10
2,TOOLBAR,82011.0,id=spreadApp,US_10
3,TOOLBAR,89845.0,id=webApp,US_10
4,TOOLBAR,92251.0,id=spreadApp,US_10


In [7]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count}회")

TOOLBAR: 5953회
BUTTON: 4740회
COMBOBOX: 1713회
START: 1350회
KEYPRESS: 768회
MENU: 536회
MENUITEM: 202회
RADIO_BTN: 182회
GET_HELP: 137회
TEXTBOX_ONFOCUS: 113회
TEXTBOX_KILLFOCUS → BUTTON: 56회
KEYPRESS → COMBOBOX: 47회
TEXTBOX_KILLFOCUS: 43회
KEYPRESS → KEYPRESS: 38회
TEXTBOX_KILLFOCUS → TOOLBAR: 8회
KEYPRESS → COMBOBOX → COMBOBOX: 7회
TOOLBAR → TOOLBAR: 6회
TEXTBOX_KILLFOCUS → MENU: 4회
COMBOBOX → BUTTON: 3회
BREAKOFF: 3회
RESTART: 3회
TEXTBOX_KILLFOCUS → TEXTBOX_ONFOCUS: 3회
KEYPRESS → COMBOBOX → COMBOBOX → COMBOBOX: 2회
TEXTBOX_KILLFOCUS → MENUITEM: 2회
KEYPRESS → KEYPRESS → COMBOBOX: 1회
KEYPRESS → COMBOBOX → KEYPRESS → COMBOBOX: 1회
MENU → MENU: 1회
COMBOBOX → COMBOBOX: 1회


### PS1_4

In [ ]:
# 사용 예시
problem_num = 'ps1_4'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## 문제 공통 행동 ##
# 최종 제출 프로세스 처리 - BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})

## ps1_4 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'ENVIRONMENT'])].copy()


pattern_map = {
    ('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON',
    ('BREAKOFF', 'END') : 'BREAKOFF'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

처리된 데이터 크기: 13929 행
제거된 행 수: 0


,event_type,timestamp,event_description,SEQID
0,START,0.0,TEST_TIME=358,US_10
1,TEXTLINK,76161.0,id=u06a_default_txt3|*$href=u06a_popup1|*$targ...,US_10
2,BUTTON,79561.0,id=u06a_popup1_txt4,US_10
3,RADIO_BTN,93553.0,id=u06arg1_prop1,US_10
4,RADIO_BTN,117389.0,id=u06arg2_prop4,US_10


In [9]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count}회")

RADIO_BTN: 6466회
BUTTON: 4061회
START: 1347회
TEXTLINK: 1341회
TOOLBAR: 430회
TEXTBOX_ONFOCUS: 86회
TEXTBOX_KILLFOCUS: 51회
MENU: 34회
GET_HELP: 23회
TEXTBOX_KILLFOCUS → TEXTLINK: 16회
TEXTBOX_ONFOCUS → TEXTBOX_KILLFOCUS: 7회
TEXTBOX_KILLFOCUS → TOOLBAR: 7회
TEXTBOX_KILLFOCUS → BUTTON: 6회
TEXTBOX_KILLFOCUS → RADIO_BTN: 4회
BREAKOFF: 1회
BUTTON → TEXTLINK: 1회
TEXTBOX_KILLFOCUS → TOOLBAR → TEXTBOX_ONFOCUS: 1회
TEXTBOX_KILLFOCUS → GET_HELP: 1회
TEXTBOX_KILLFOCUS → MENU: 1회


### PS1_5

In [ ]:
# 사용 예시
problem_num = 'ps1_5'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## 문제 공통 행동 ##
# 최종 제출 프로세스 처리 - BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})

## ps1_5 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'ENVIRONMENT','HISTORY_NEXT','HISTORY_BACK','HISTORY_ADD',
                                    'COPY', 'BOOKMARK_ADD'])].copy()


pattern_map = {
    ('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON',
    ('BREAKOFF', 'END') : 'BREAKOFF',
    ('COMBOBOX', 'INFORMATION', 'GLOBAL_VAR', 'TRANSLATION'): 'COMBOBOX'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

처리된 데이터 크기: 19829 행
제거된 행 수: 20


,event_type,timestamp,event_description,SEQID
0,START,0.0,TEST_TIME=384,US_10
1,COMBOBOX,16837.0,id=inquiry_1_interaction_ddmenupopup|$*action=...,US_10
2,COMBOBOX,54268.0,id=inquiry_1_interaction_ddmenupopup|$*index=2,US_10
3,COMBOBOX,54525.0,id=inquiry_1_interaction_ddmenupopup|$*action=...,US_10
4,BUTTON,55976.0,id=next,US_10


In [11]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count}회")

TEXTLINK: 5431회
TOOLBAR: 5413회
COMBOBOX: 4732회
BUTTON: 2812회
START: 1346회
KEYPRESS: 38회
TEXTBOX_ONFOCUS: 18회
MENU: 13회
TEXTBOX_KILLFOCUS: 8회
TEXTBOX_KILLFOCUS → BUTTON: 5회
TEXTBOX_KILLFOCUS → TEXTLINK: 4회
GET_HELP: 3회
RESTART: 2회
MENUITEM: 2회
KEYPRESS → COMBOBOX: 1회
TEXTBOX_KILLFOCUS → TOOLBAR: 1회
KEYPRESS → KEYPRESS → KEYPRESS: 1회
KEYPRESS → KEYPRESS: 1회
TOOLBAR → COMBOBOX: 1회
TOOLBAR → TOOLBAR: 1회


### PS1_6

In [ ]:
# 사용 예시
problem_num = 'ps1_6'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## 문제 공통 행동 ##
# 최종 제출 프로세스 처리 - BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})


## Translation type, description
pattern = ('HISTORY_ADD', 'BOX_PRESS')
data = replace_pattern(data, pattern, 'TEXTLINK', 'id=u021_404')


## ps1_5 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'HISTORY_NEXT', 'HISTORY_BACK', 'HISTORY_ADD', 'BOX_PRESS'])].copy()


pattern_map = {
    ('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON',
    ('BREAKOFF', 'END') : 'BREAKOFF'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

처리된 데이터 크기: 29437 행
제거된 행 수: 39


,event_type,timestamp,event_description,SEQID
0,START,0.0,TEST_TIME=346,US_10
1,TEXTLINK,21752.0,id=u021_default_txt6|*$href=unit21page4|*$targ...,US_10
2,COMBOBOX,52883.0,id=u021_pg4_menu1|*$index=7,US_10
3,COMBOBOX,59790.0,id=u021_pg4_menu2|*$index=2,US_10
4,BUTTON,61461.0,id=u021_pg4_txt23,US_10


In [13]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count}회")

BUTTON: 9401회
TAB: 6835회
COMBOBOX: 5792회
CHECKBOX: 4248회
START: 1346회
TOOLBAR: 869회
TEXTLINK: 693회
KEYPRESS: 103회
TEXTBOX_ONFOCUS: 26회
KEYPRESS → COMBOBOX: 21회
TEXTBOX_KILLFOCUS → BUTTON: 20회
MENU: 18회
GET_HELP: 13회
COMBOBOX → BUTTON: 7회
TEXTBOX_KILLFOCUS: 6회
KEYPRESS → KEYPRESS → COMBOBOX: 5회
RESTART: 3회
TAB → TEXTLINK: 2회
KEYPRESS → KEYPRESS → KEYPRESS → COMBOBOX: 1회
TEXTBOX_KILLFOCUS → TEXTBOX_ONFOCUS: 1회
TAB → TAB: 1회


### PS1_7

In [ ]:
# 사용 예시
problem_num = 'ps1_7'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## 문제 공통 행동 ##
# 최종 제출 프로세스 처리 - BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})


## ps1_7 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'ENVIRONMENT', 'SPLITSCREEN','MC_HELP_TOOLBAR','MC_HELP_MENUITEM',
                                    'COPY', 'PASTE'])].copy()


pattern_map = {
    ('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON',
    ('BREAKOFF', 'END') : 'BREAKOFF'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

처리된 데이터 크기: 70412 행
제거된 행 수: 3441


,event_type,timestamp,event_description,SEQID
0,START,0.0,TEST_TIME=386,US_10
1,TOOLBAR,35145.0,id=spreadApp,US_10
2,BUTTON,39465.0,id=next,US_10
3,BUTTON,41166.0,id=ok,US_10
4,START,0.0,TEST_TIME=387,US_1000


In [15]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count}회")

TEXTBOX_ONFOCUS: 10767회
KEYPRESS: 9221회
COMBOBOX: 7702회
TOOLBAR: 7595회
TEXTBOX_KILLFOCUS → TOOLBAR: 6188회
TEXTBOX_KILLFOCUS → TEXTBOX_ONFOCUS: 4829회
TEXTBOX_KILLFOCUS: 3537회
TEXTBOX_KILLFOCUS → TOOLBAR → TEXTBOX_ONFOCUS: 1757회
BUTTON: 1743회
START: 1346회
TEXTBOX_KILLFOCUS → BUTTON: 954회
KEYPRESS → TEXTBOX_KILLFOCUS → TEXTBOX_ONFOCUS: 305회
KEYPRESS → KEYPRESS: 161회
TEXTBOX_KILLFOCUS → BUTTON → TEXTBOX_ONFOCUS: 144회
MENU: 126회
TEXTBOX_KILLFOCUS → MENU: 77회
KEYPRESS → KEYPRESS → TEXTBOX_KILLFOCUS → TEXTBOX_ONFOCUS: 43회
TOOLBAR → TEXTBOX_ONFOCUS: 35회
GET_HELP: 32회
TEXTBOX_KILLFOCUS → MENUITEM: 24회
KEYPRESS → TEXTBOX_KILLFOCUS → TEXTBOX_ONFOCUS → TEXTBOX_KILLFOCUS → TEXTBOX_ONFOCUS: 19회
KEYPRESS → TEXTBOX_KILLFOCUS → TEXTBOX_ONFOCUS → KEYPRESS: 18회
TEXTBOX_ONFOCUS → TEXTBOX_KILLFOCUS: 18회
TEXTBOX_KILLFOCUS → MENU → TEXTBOX_ONFOCUS: 13회
MENUITEM: 13회
KEYPRESS → TEXTBOX_KILLFOCUS: 10회
KEYPRESS → COMBOBOX: 9회
TEXTBOX_KILLFOCUS → GET_HELP: 8회
TOOLBAR → TEXTBOX_ONFOCUS → TEXTBOX_KILLFOCUS: 6회
KEY

### PS2_1

In [ ]:
# 사용 예시
problem_num = 'ps2_1'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## 문제 공통 행동 ##
# 최종 제출 프로세스 처리 - BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})


## ps2_1 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'ENVIRONMENT', 'COPY', 'PASTE', 'MC_HELP_MENUITEM',
                                    'SS_SEARCH', 'SS_SORT'])].copy()


pattern_map = {
    ('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON',
    ('BREAKOFF', 'END') : 'BREAKOFF'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

처리된 데이터 크기: 26060 행
제거된 행 수: 4652


,event_type,timestamp,event_description,SEQID
0,START,1.0,TEST_TIME=444,US_1004
1,CELL_CHANGE,67909.0,id=content_spreadsheet_ColaD_row65,US_1004
2,TOOLBAR,75203.0,id=mailApp,US_1004
3,GET_HELP,102817.0,REQUEST,US_1004
4,TEXTBOX_ONFOCUS,110726.0,id=u019_email_message|*$value=,US_1004


In [17]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count}회")

KEYPRESS: 4812회
TEXTBOX_ONFOCUS: 4731회
TOOLBAR: 3378회
TEXTBOX_KILLFOCUS → BUTTON: 1927회
BUTTON: 1826회
TEXTBOX_KILLFOCUS → TOOLBAR: 1488회
START: 1355회
CELL_CHANGE: 1032회
TEXTBOX_KILLFOCUS: 828회
MENU: 656회
KEYPRESS → KEYPRESS: 644회
TEXTBOX_KILLFOCUS → MENU: 317회
TEXTBOX_KILLFOCUS → TEXTBOX_ONFOCUS: 301회
MENUITEM: 214회
TEXTBOX_KILLFOCUS → TOOLBAR → TEXTBOX_ONFOCUS: 213회
TEXTBOX_KILLFOCUS → BUTTON → TEXTBOX_ONFOCUS: 196회
COMBOBOX: 193회
TEXTBOX_KILLFOCUS → MENUITEM: 154회
RADIO_BTN: 116회
KEYPRESS → KEYPRESS → KEYPRESS: 63회
TEXTBOX_KILLFOCUS → MENUITEM → TEXTBOX_ONFOCUS: 54회
GET_HELP: 42회
TEXTBOX_ONFOCUS → TEXTBOX_KILLFOCUS: 38회
TEXTBOX_KILLFOCUS → MENU → TEXTBOX_ONFOCUS: 38회
TEXTBOX_KILLFOCUS → GET_HELP: 14회
TEXTBOX_KILLFOCUS → CELL_CHANGE: 6회
TOOLBAR → TEXTBOX_ONFOCUS: 4회
KEYPRESS → TOOLBAR: 4회
KEYPRESS → TEXTBOX_KILLFOCUS: 3회
MENUITEM → TEXTBOX_ONFOCUS: 2회
KEYPRESS → TEXTBOX_ONFOCUS: 2회
RESTART: 2회
TEXTBOX_KILLFOCUS → GET_HELP → TEXTBOX_ONFOCUS: 2회
TEXTBOX_ONFOCUS → TEXTBOX_KILLFOCUS → TOO

### PS2_2

In [ ]:
# 사용 예시
problem_num = 'ps2_2'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## 문제 공통 행동 ##
# 최종 제출 프로세스 처리 - BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})


## ps2_2 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'ENVIRONMENT', 'SS_SEARCH', 'SS_SORT'])].copy()


pattern_map = {
    ('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

처리된 데이터 크기: 28310 행
제거된 행 수: 0


,event_type,timestamp,event_description,SEQID
0,START,0.0,TEST_TIME=337,US_1004
1,TOOLBAR,38125.0,id=spreadApp,US_1004
2,MENU,45055.0,id=ss-data-menu,US_1004
3,MENU,49045.0,id=ss-data-menu,US_1004
4,MENUITEM,49887.0,key=sort,US_1004


In [19]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count}회")

CELL_CHANGE: 11970회
TOOLBAR: 7998회
BUTTON: 3733회
START: 1355회
COMBOBOX: 928회
MENU: 678회
RADIO_BTN: 651회
MENUITEM: 343회
TEXTBOX_ONFOCUS: 224회
TEXTBOX_KILLFOCUS → BUTTON: 108회
TEXTBOX_KILLFOCUS: 82회
GET_HELP: 34회
TEXTBOX_KILLFOCUS → TOOLBAR: 29회
TEXTBOX_KILLFOCUS → TEXTBOX_ONFOCUS: 11회
TEXTBOX_KILLFOCUS → MENU: 4회
TOOLBAR → TOOLBAR: 2회
TEXTBOX_ONFOCUS → TEXTBOX_KILLFOCUS: 2회
TEXTBOX_KILLFOCUS → RADIO_BTN: 1회


### PS2_3

In [ ]:
# 사용 예시
problem_num = 'ps2_3'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## 문제 공통 행동 ##
# 최종 제출 프로세스 처리 - BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})


## ps2_3 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'ENVIRONMENT','SS_SEARCH', 'SS_SORT', 'HISTORY_ADD','HISTORY_BACK','HISTORY_NEXT',
                                    'BOOKMARK_ADD', 'BUYBOOK'])].copy()


pattern_map = {
    ('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

처리된 데이터 크기: 23455 행
제거된 행 수: 94


,event_type,timestamp,event_description,SEQID
0,START,0.0,TEST_TIME=331,US_1004
1,TEXTLINK,42555.0,id=u07_default_txt3|*$href=unit7page1|*$target...,US_1004
2,TOOLBAR,55938.0,id=toolbar_back_btn,US_1004
3,TEXTLINK,65689.0,id=u07_default_txt3|*$href=unit7page1|*$target...,US_1004
4,TEXTLINK,69588.0,id=u07_pg1_txt7|*$href=u07_pg1_popup1|*$target...,US_1004


In [21]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count}회")

TEXTLINK: 8123회
BUTTON: 7173회
TOOLBAR: 6572회
START: 1355회
KEYPRESS: 108회
TEXTBOX_ONFOCUS: 63회
TEXTBOX_KILLFOCUS: 28회
MENU: 21회
TEXTBOX_KILLFOCUS → TEXTLINK: 20회
TEXTBOX_KILLFOCUS → BUTTON: 11회
KEYPRESS → KEYPRESS: 7회
TEXTBOX_KILLFOCUS → TOOLBAR: 4회
BUTTON → BUTTON: 2회
TEXTBOX_KILLFOCUS → TEXTBOX_ONFOCUS: 2회
TOOLBAR → TOOLBAR: 2회
MENUITEM: 2회
RESTART: 2회
BUTTON → TEXTLINK: 1회
TEXTBOX_KILLFOCUS → TOOLBAR → TEXTBOX_ONFOCUS: 1회
GET_HELP: 1회


### PS2_4

In [ ]:
# 사용 예시
problem_num = 'ps2_4'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## 문제 공통 행동 ##
# 최종 제출 프로세스 처리 - BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})


## Translation type, description
patterns = [
    (('BUTTON', 'SUBMIT_RESERVATION_SUCCESS'), 'SUCCESS', 'id=submit'),
    (('BUTTON', 'SUBMIT_RESERVATION_FAILURE'), 'FAILURE', 'id=submit'),
    (('BUTTON', 'CHANGE_RESERVATION_SUCCESS'), 'SUCCESS', 'id=change'),
    (('BUTTON', 'CHANGE_RESERVATION_FAILURE'), 'FAILURE', 'id=change')
]

for pattern, new_type, new_desc in patterns:
    data = replace_pattern(data, pattern, new_type, new_desc)


## ps2_4 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'ENVIRONMENT', 'MC_HELP_TOOLBAR', 'BOOKMARK_TOOLBAR', 'COPY', 'HOME_TOOLBAR',
                                    'HISTORY_ADD', 'HISTORY_BACK', 'HISTORY_BACK_TOOLBAR', 'HISTORY_FORWARD_TOOLBAR', 'HISTORY_NEXT',
                                    'WP_HELP_MENUITEM', 'MC_HELP_MENUITEM', 'NEW_FOLDER', 'MC_SORT', 'BOOKMARK_ADD',
                                    'SORT_MENU', 'MAIL_COPIED', 'WB_HELP_TOOLBAR', 'WB_HELP_MENUITEM',
                                    'DELETE_FOLDER', 'COPY_MENUITEM', 'PASTE_MENUITEM', 'PASTE'])].copy()


pattern_map = {
    ('FOLDER_VIEWED', 'MAIL_DROP', 'MAIL_MOVED'): 'MAIL_DROP',
    ('FOLDER_VIEWED', 'MAIL_MOVED', 'MAIL_DROP'): 'MAIL_DROP',
    ('BUTTON', 'MAIL_MOVED'): 'BUTTON',
    ('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON',
    ('BREAKOFF', 'END') : 'BREAKOFF'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

처리된 데이터 크기: 49582 행
제거된 행 수: 1560


,event_type,timestamp,event_description,SEQID
0,START,0.0,TEST_TIME=320,US_1004
1,FOLDER_VIEWED,18726.0,id=ReservationsFolder,US_1004
2,FOLDER_VIEWED,20299.0,id=ReservationsFolder,US_1004
3,FOLDER_UNFOLDED,24930.0,id=MtgRoomFolder,US_1004
4,FOLDER_FOLDED,26607.0,id=MtgRoomFolder,US_1004


In [23]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count}회")

TOOLBAR: 11490회
MAIL_VIEWED: 7929회
COMBOBOX: 6288회
BUTTON: 5997회
TEXTLINK: 5652회
FOLDER_VIEWED: 5344회
START: 1355회
SUCCESS: 1112회
FAILURE: 994회
KEYPRESS: 718회
MAIL_DRAG: 591회
TEXTBOX_ONFOCUS: 411회
KEYPRESS → KEYPRESS: 372회
MAIL_DROP: 370회
MENU: 276회
TEXTBOX_KILLFOCUS → TOOLBAR: 174회
FOLDER_UNFOLDED: 160회
FOLDER_FOLDED: 139회
MAIL_VIEWED → MAIL_DROP: 99회
TEXTBOX_KILLFOCUS: 99회
MENUITEM: 69회
KEYPRESS → KEYPRESS → KEYPRESS: 67회
TEXTBOX_KILLFOCUS → BUTTON: 49회
TEXTBOX_KILLFOCUS → TOOLBAR → TEXTBOX_ONFOCUS: 40회
TEXTBOX_KILLFOCUS → MAIL_VIEWED: 29회
MAIL_VIEWED → FOLDER_VIEWED: 25회
GET_HELP: 24회
TEXTBOX_KILLFOCUS → TEXTLINK: 22회
TEXTBOX_KILLFOCUS → MENU: 18회
MAIL_VIEWED → MAIL_VIEWED: 13회
TEXTBOX_KILLFOCUS → TEXTBOX_ONFOCUS: 11회
TEXTBOX_KILLFOCUS → FOLDER_VIEWED: 11회
TEXTBOX_KILLFOCUS → MENUITEM: 6회
KEYPRESS → COMBOBOX: 6회
RESTART: 5회
TEXTBOX_ONFOCUS → TEXTBOX_KILLFOCUS: 5회
KEYPRESS → KEYPRESS → KEYPRESS → KEYPRESS: 4회
TEXTBOX_KILLFOCUS → MENU → TEXTBOX_ONFOCUS: 4회
TEXTBOX_KILLFOCUS → TOOLBAR 

### PS2_5

In [ ]:
# 사용 예시
problem_num = 'ps2_5'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## 문제 공통 행동 ##
# 최종 제출 프로세스 처리 - BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})


## ps2_5 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'MC_HELP_TOOLBAR', 'MAIL_SENT', 'PASTE', 'COPY', 'SORT_MENU',
                                    'MC_HELP_MENUITEM', 'MC_SORT', 'NEW_FOLDER', 'TRANSLATION', 'MAIL_DELETED'])].copy()


pattern_map = {
    ('FOLDER_VIEWED', 'MAIL_DROP', 'MAIL_MOVED'): 'MAIL_DROP',
    ('FOLDER_VIEWED', 'MAIL_MOVED', 'MAIL_DROP'): 'MAIL_DROP',
    ('BUTTON', 'MAIL_MOVED'): 'BUTTON',
    ('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON',
    ('BREAKOFF', 'END') : 'BREAKOFF'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

처리된 데이터 크기: 31396 행
제거된 행 수: 49816


,event_type,timestamp,event_description,SEQID
0,START,0.0,TEST_TIME=1067,US_1004
1,MENU,13123.0,id=file-menu,US_1004
2,MENUITEM,19690.0,key=reply,US_1004
3,TEXTBOX_ONFOCUS,25549.0,id=email_message|*$value= ......................,US_1004
4,KEYPRESS,29999.0,count=23,US_1004


In [25]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count}회")

KEYPRESS: 27279회
KEYPRESS → KEYPRESS: 8931회
TEXTBOX_ONFOCUS: 4455회
MAIL_VIEWED: 4080회
TOOLBAR: 2748회
BUTTON: 2546회
KEYPRESS → KEYPRESS → KEYPRESS: 2227회
TEXTBOX_KILLFOCUS → TOOLBAR: 1674회
START: 1354회
TEXTBOX_KILLFOCUS: 1259회
FOLDER_VIEWED: 1147회
TEXTBOX_KILLFOCUS → TEXTBOX_ONFOCUS: 910회
MENU: 693회
TEXTBOX_KILLFOCUS → MENU: 599회
MENUITEM: 465회
TEXTBOX_KILLFOCUS → BUTTON: 446회
TEXTBOX_KILLFOCUS → MENUITEM: 332회
TEXTBOX_KILLFOCUS → TOOLBAR → TEXTBOX_ONFOCUS: 144회
MAIL_DRAG: 132회
TEXTBOX_KILLFOCUS → MENU → TEXTBOX_ONFOCUS: 116회
TEXTBOX_KILLFOCUS → MENUITEM → TEXTBOX_ONFOCUS: 111회
KEYPRESS → KEYPRESS → KEYPRESS → KEYPRESS: 89회
MAIL_DROP: 55회
TEXTBOX_KILLFOCUS → MENUITEM → TEXTBOX_ONFOCUS → TEXTBOX_KILLFOCUS: 48회
KEYPRESS → TEXTBOX_KILLFOCUS: 44회
TEXTBOX_KILLFOCUS → BUTTON → TEXTBOX_ONFOCUS: 43회
TEXTBOX_ONFOCUS → TEXTBOX_KILLFOCUS: 35회
TEXTBOX_KILLFOCUS → MAIL_VIEWED: 35회
TEXTBOX_KILLFOCUS → FOLDER_VIEWED: 27회
KEYPRESS → TEXTBOX_ONFOCUS: 21회
KEYPRESS → KEYPRESS → TEXTBOX_KILLFOCUS: 19회
KEYP

### PS2_6

In [ ]:
# 사용 예시
problem_num = 'ps2_6'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## 문제 공통 행동 ##
# 최종 제출 프로세스 처리 - BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})


## ps2_6 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'SORT_MENU', 'MC_HELP_MENUITEM', 'PASTE', 'COPY', 'MC_SORT', 'DELETE_FOLDER',
                                    'MAIL_COPIED', 'MAIL_SENT', 'NEW_FOLDER', 'MAIL_DELETED', 'MC_HELP_TOOLBAR'])].copy()


pattern_map = {
('FOLDER_VIEWED', 'MAIL_DROP', 'MAIL_MOVED'): 'MAIL_DROP',
('FOLDER_VIEWED', 'MAIL_MOVED', 'MAIL_DROP'): 'MAIL_DROP',
('BUTTON', 'MAIL_MOVED'): 'BUTTON',
('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

처리된 데이터 크기: 32364 행
제거된 행 수: 1115


,event_type,timestamp,event_description,SEQID
0,START,0.0,TEST_TIME=344,US_1004
1,MENU,12127.0,id=edit-menu,US_1004
2,MENUITEM,17783.0,key=sort-mc,US_1004
3,RADIO_BTN,22772.0,id=prioritysortmcdesc,US_1004
4,BUTTON,24489.0,id=sortValidation,US_1004


In [27]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count}회")

MAIL_VIEWED: 7098회
FOLDER_VIEWED: 5736회
MAIL_DRAG: 4867회
BUTTON: 3897회
MAIL_DROP: 3013회
MAIL_VIEWED → MAIL_DROP: 1433회
START: 1353회
TOOLBAR: 1008회
KEYPRESS: 745회
MENU: 560회
MENUITEM: 435회
TEXTBOX_ONFOCUS: 277회
KEYPRESS → KEYPRESS: 227회
MAIL_VIEWED → FOLDER_VIEWED: 178회
TEXTBOX_KILLFOCUS → BUTTON: 99회
TEXTBOX_KILLFOCUS: 96회
KEYPRESS → KEYPRESS → KEYPRESS: 50회
TEXTBOX_KILLFOCUS → FOLDER_VIEWED: 49회
FOLDER_UNFOLDED: 42회
FOLDER_FOLDED: 31회
TEXTBOX_KILLFOCUS → MENU: 12회
KEYPRESS → MAIL_VIEWED: 11회
TEXTBOX_KILLFOCUS → MENUITEM: 10회
MAIL_VIEWED → MAIL_VIEWED: 9회
TEXTBOX_KILLFOCUS → TOOLBAR: 8회
TEXTBOX_KILLFOCUS → TEXTBOX_ONFOCUS: 5회
RADIO_BTN: 4회
GET_HELP: 3회
TEXTBOX_KILLFOCUS → MENU → TEXTBOX_ONFOCUS: 2회
KEYPRESS → FOLDER_VIEWED: 2회
MENUITEM → FOLDER_VIEWED: 2회
MENU → MENU: 2회
KEYPRESS → FOLDER_VIEWED → FOLDER_VIEWED: 2회
KEYPRESS → TEXTBOX_KILLFOCUS → TEXTBOX_ONFOCUS: 2회
TEXTBOX_KILLFOCUS → MENUITEM → TEXTBOX_ONFOCUS: 1회
FOLDER_UNFOLDED → BUTTON: 1회
KEYPRESS → MAIL_VIEWED → MAIL_VIEWED → MAI

### PS2_7

In [ ]:
# 사용 예시
problem_num = 'ps2_7'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## 문제 공통 행동 ##
# 최종 제출 프로세스 처리 - BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})

# 특정 키워드 포함 시 매핑
mask = data['event_description'].str.contains('u023_pg6_popup1', na=False)

data.loc[mask, 'event_description'] = (
    data.loc[mask, 'event_description']
        .str.replace(r'(?<=id=u023_)pg5', 'pg6', regex=True)
)

## Translation ps2_7
conditions = [
    ('HISTORY_ADD', 'pageid=unit23page3', 'TEXTLINK', 'id=u023_default_txt3'),
    ('HISTORY_ADD', 'pageid=unit23page4', 'TEXTLINK', 'id=u023_default_txt4'),
    ('HISTORY_ADD', 'pageid=unit23page5', 'TEXTLINK', 'id=u023_default_txt5'),
    ('HISTORY_ADD', 'pageid=unit23page6', 'TEXTLINK', 'id=u023_default_txt6'),
    ('HISTORY_ADD', 'pageid=unit23page7', 'TEXTLINK', 'id=u023_default_txt7'),
    ('HISTORY_ADD', 'pageid=unit23page8', 'TEXTLINK', 'id=u023_default_txt8')
]

for old_type, old_desc, new_type, new_desc in conditions:
    mask = (data['event_type'] == old_type) & (data['event_description'] == old_desc)
    data.loc[mask, 'event_type'] = new_type
    data.loc[mask, 'event_description'] = new_desc


## ps2_7 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'ENVIRONMENT', 'MAIL_SENT', 'HISTORY_BACK', 'COPY', 'SORT_MENU', 'MC_SORT', 'HISTORY_ADD',
                                    'HISTORY_NEXT', 'MC_HELP_MENUITEM', 'PASTE', 'MC_HELP_TOOLBAR', 'BOOKMARK_ADD'])].copy()

pattern_map = {
('FOLDER_VIEWED', 'MAIL_DROP', 'MAIL_MOVED'): 'MAIL_DROP',
('FOLDER_VIEWED', 'MAIL_MOVED', 'MAIL_DROP'): 'MAIL_DROP',
('BUTTON', 'MAIL_MOVED'): 'BUTTON',
('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON',
('BREAKOFF', 'END') : 'BREAKOFF'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

처리된 데이터 크기: 26108 행
제거된 행 수: 2796


,event_type,timestamp,event_description,SEQID
0,START,0.0,TEST_TIME=6084,US_1004
1,TEXTLINK,13464.0,id=u023_default_txt5,US_1004
2,TOOLBAR,27636.0,id=toolbar_back_btn,US_1004
3,TEXTLINK,31915.0,id=u023_default_txt10|*$href=unit23page1|*$tar...,US_1004
4,TEXTLINK,39312.0,id=u023_pg1_txt9|*$href=unit23page12|*$target=...,US_1004


In [4]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count}회")

BUTTON: 6352회
TEXTLINK: 4784회
TOOLBAR: 4752회
KEYPRESS: 1967회
MAIL_VIEWED: 1939회
START: 1353회
TEXTBOX_ONFOCUS: 1120회
FOLDER_VIEWED: 1116회
KEYPRESS → KEYPRESS: 605회
COMBOBOX: 573회
RADIO_BTN: 562회
TEXTBOX_KILLFOCUS → BUTTON: 445회
TEXTBOX_KILLFOCUS: 265회
TEXTBOX_KILLFOCUS → TOOLBAR: 219회
MENU: 133회
KEYPRESS → KEYPRESS → KEYPRESS: 75회
TEXTBOX_KILLFOCUS → TEXTLINK: 69회
MENUITEM: 66회
TEXTBOX_KILLFOCUS → TOOLBAR → TEXTBOX_ONFOCUS: 60회
KEYPRESS → MAIL_VIEWED: 51회
TEXTBOX_KILLFOCUS → MENU: 48회
TEXTBOX_KILLFOCUS → TEXTBOX_ONFOCUS: 30회
TEXTBOX_KILLFOCUS → RADIO_BTN: 27회
TEXTBOX_KILLFOCUS → MENUITEM → TEXTBOX_ONFOCUS: 25회
MAIL_DRAG: 24회
KEYPRESS → MAIL_VIEWED → MAIL_VIEWED: 24회
KEYPRESS → MAIL_VIEWED → MAIL_VIEWED → MAIL_VIEWED: 14회
TEXTBOX_KILLFOCUS → MENUITEM: 13회
TEXTBOX_KILLFOCUS → MAIL_VIEWED: 12회
TEXTBOX_KILLFOCUS → MENUITEM → TEXTBOX_ONFOCUS → TEXTBOX_KILLFOCUS: 9회
TEXTBOX_KILLFOCUS → MENU → TEXTBOX_ONFOCUS: 9회
MAIL_DROP: 8회
TEXTBOX_KILLFOCUS → FOLDER_VIEWED: 7회
GET_HELP: 6회
TEXTBOX_KILLFOCU

### 2, 3차 전처리 방법론

In [ ]:
# 미국만 존재
remove_vars_dict = {
    'ps1_1' : {'test_time', 'end', 'value'},
    'ps1_2' : {'test_time', 'end', 'value'},
    'ps1_3' : {'test_time', 'end', 'value'},
    'ps1_4' : {'test_time', 'end', 'value', 'href', 'target'},
    'ps1_5' : {'test_time', 'end', 'value', 'href'},
    'ps1_6' : {'test_time', 'end', 'value', 'href', 'target'},
    'ps1_7' : {'test_time', 'end', 'value'},
    'ps2_1' : {'test_time', 'end', 'value'},
    'ps2_2' : {'test_time', 'end', 'value'},
    'ps2_3' : {'test_time', 'end', 'value', 'href'},
    'ps2_4' : {'test_time', 'end', 'value', 'href', 'target'},
    'ps2_5' : {'test_time', 'end', 'value'},
    'ps2_6' : {'test_time', 'end', 'value'},
    'ps2_7' : {'test_time', 'end', 'value', 'href'}
}

def process_event_description(description):
    """
    event_description을 특수문자로 분리하고 필터링하는 함수
    
    Args:
        description (str): 원본 event_description
    
    Returns:
        list: 필터링된 구성요소 리스트
    """
    
    # 특수문자로 분리 ('|', '*', ')
    components = re.split(r'[|*$]', str(description))
    
    # 빈 문자열 제거
    components = [comp.strip() for comp in components if comp.strip()]
    
    # 제거할 시스템변수 목록 - 다양한 값 추가 가능
    remove_vars = remove_vars_dict[problem_num]
    
    # 추가로 제거할 상세정보 목록도 생성할 수 있음
    remove_details = {'nan', ',', '.'}
    
    # 필터링된 구성요소 리스트
    filtered_components = []
    
    for comp in components:
        if '=' in comp or comp == 'end':
            # '=' 앞의 시스템변수 추출
            var_name = comp.split('=')[0].strip()
            
            # 제거 대상이 아니면 리스트에 추가
            if var_name not in remove_vars:
                filtered_components.append(comp)
        else:
            # '='가 없는 경우도 포함
            if not any(k in comp for k in remove_details):
                filtered_components.append(comp)
    
    fin_string = '|'.join(filtered_components)
    
    return fin_string

## create_event_type_dict(df) 수정 요망
def create_event_type_dict(df):
    """
    event_type과 event_desc_list를 기반으로 네 개의 딕셔너리를 생성하는 함수
    
    Parameters:
    df (pd.DataFrame): event_type과 event_desc_list 컬럼을 포함한 데이터프레임
    
    Returns:
    tuple: (unit_dict, unit_count_dict, token_dict, token_count_dict)
    """
    
    # 결과를 저장할 딕셔너리들
    event_dict = defaultdict(set)  # 고유 unit을 위한 딕셔너리
    event_count_dict = defaultdict(lambda: defaultdict(int))  # unit 카운트
    token_dict = defaultdict(set)  # 고유 token을 위한 딕셔너리
    token_count_dict = defaultdict(lambda: defaultdict(int))  # token 카운트
    
    # 데이터프레임을 순회하며 처리
    for _, row in df.iterrows():
        event_type = row['event_type']
        event_desc = row['event_desc_list']
        
        # unit (전체 event_desc) 처리
        event_dict[event_type].add(event_desc)
        event_count_dict[event_type][event_desc] += 1
        
        # token (|로 구분된 각 부분) 처리
        tokens = event_desc.split('|')
        
        for token in tokens:
            token_dict[event_type].add(token)  # update() 대신 add() 사용
            token_count_dict[event_type][token] += 1
    
    # set을 list로 변환
    unit_dict = {key: list(value) for key, value in event_dict.items()}
    token_unique_dict = {key: list(value) for key, value in token_dict.items()}
    
    # defaultdict를 일반 dict로 변환
    unit_count_dict = {key: dict(value) for key, value in event_count_dict.items()}
    token_count_dict_final = {key: dict(value) for key, value in token_count_dict.items()}
    
    return unit_dict, unit_count_dict, token_unique_dict, token_count_dict_final

In [ ]:
## 1번 문제에 해당하는 데이터 불러오기
## 전체 조합 확인 후 하나의 딕셔너리로 정리
## unit, token 조합 확인 후 제거할 목록 확인
## 제거 후 하나의 파일로 저장
## 2번 파일에 대해서도 동일하게 수행
## -> 필요한 기능, unit, token 조합의 전체 딕셔너리 생성

unit_dict_list = []
unit_cnt_list = []
token_dict_list = []
token_cnt_list = []

left = range(1, 3)   # 1 ~ 2
right = range(1, 8)  # 1 ~ 7
pairs = product(left, right)

# 병합할 딕셔너리 초기화
merged_unit_dict = defaultdict(set)
merged_token_dict = defaultdict(set)
merged_token_cnt = defaultdict(lambda: defaultdict(int))
merged_unit_cnt = defaultdict(lambda: defaultdict(int))

for pair in pairs:
    i, j = pair
    
    problem_num = f'ps{i}_{j}'
    
    path_1st = f'input_data/1st_data/us_{problem_num}.pkl'
    path_2nd = f'input_data/2nd_data/us_{problem_num}.pkl'
    
    fin_data = pd.read_pickle(path_1st)
    
    fin_data.event_type = fin_data.event_type.str.lower()
    fin_data.event_description = fin_data.event_description.str.lower()
    
    fin_data['event_desc_list'] = fin_data['event_description'].apply(process_event_description).copy()
    
    unit_dict, unit_cnt, token_dict, token_cnt = create_event_type_dict(fin_data)

    unit_dict_list.append(unit_dict)
    unit_cnt_list.append(unit_cnt)
    token_dict_list.append(token_dict)
    token_cnt_list.append(token_cnt)
    
    fin_data.to_pickle(path_2nd)
    
    # ===== 각 문제별로 딕셔너리 저장 =====
    token_unit_path = 'input_data/token_units/'
    
    # 각 문제별 파일명
    token_name_individual = f'us_token_{problem_num}.pkl'
    unit_name_individual = f'us_unit_{problem_num}.pkl'
    token_cnt_individual = f'us_token_cnt_{problem_num}.pkl'
    unit_cnt_individual = f'us_unit_cnt_{problem_num}.pkl'
    
    # 각 문제별로 저장
    with open(token_unit_path + token_name_individual, "wb") as f:
        pickle.dump(token_dict, f)
    
    with open(token_unit_path + unit_name_individual, "wb") as f:
        pickle.dump(unit_dict, f)
    
    with open(token_unit_path + token_cnt_individual, "wb") as f:
        pickle.dump(token_cnt, f)
    
    with open(token_unit_path + unit_cnt_individual, "wb") as f:
        pickle.dump(unit_cnt, f)
    # ======================================
    

# unit_dict_list 병합
for d in unit_dict_list:
    for category, unit_list in d.items():
        merged_unit_dict[category].update(unit_list)  # set에 리스트의 모든 요소 추가

# token_dict_list 병합
for d in token_dict_list:
    for category, token_list in d.items():
        merged_token_dict[category].update(token_list)  # set에 리스트의 모든 요소 추가

for d in token_cnt_list:
    for category, subdict in d.items():
        for key, count in subdict.items():
            merged_token_cnt[category][key] += count  # 동일 id면 count 누적

for d in unit_cnt_list:
    for category, subdict in d.items():
        for key, count in subdict.items():
            merged_unit_cnt[category][key] += count  # 동일 id면 count 누적


# set을 list로 변환
merged_token_dict = {cat: list(tokens) for cat, tokens in merged_token_dict.items()}
merged_unit_dict = {cat: list(units) for cat, units in merged_unit_dict.items()}
merged_token_cnt = {cat: dict(ids) for cat, ids in merged_token_cnt.items()}
merged_unit_cnt = {cat: dict(ids) for cat, ids in merged_unit_cnt.items()}

token_unit_path = './input_data/token_units/'

token_name = 'us_token_ps_N.pkl'
unit_name = 'us_unit_ps_N.pkl'
token_cnt_path = 'us_token_cnt_ps_N.pkl'
unit_cnt_path = 'us_unit_cnt_ps_N.pkl'

with open(token_unit_path + token_name, "wb") as f:
    pickle.dump(merged_token_dict, f)

with open(token_unit_path + unit_name, "wb") as f:
    pickle.dump(merged_unit_dict, f)
    
with open(token_unit_path + token_cnt_path, "wb") as f:
    pickle.dump(merged_token_cnt, f)

with open(token_unit_path + unit_cnt_path, "wb") as f:
    pickle.dump(merged_unit_cnt, f)


In [ ]:
# Action 정리
maintain_list = ['mail_drag', 'mail_drop', 'folder_viewed', 'mail_viewed', 'menuitem',
                'get_help', 'restart', 'folder_unfolded', 'folder_folded', 'radio_btn',
                'breakoff', 'shortcut', 'checkbox', 'tab', 'success', 'failure']
LLM_list = ['toolbar', 'menu', 'button', 'textbox_onfocus', 'textbox_killfocus', 'combobox', 'textlink']

def parse_text_table(text_path: str) -> pd.DataFrame:
    """
    텍스트 파일의 테이블을 파싱해서 데이터프레임으로 변환
    
    Args:
        text_path: 텍스트 파일 경로
    
    Returns:
        pd.DataFrame: event_type, description, substitute, step2_output, fin_result 컬럼을 가진 데이터프레임
    """
    
    with open(text_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # 테이블 라인들 추출
    table_lines = []
    for line in content.split('\n'):
        line = line.strip()
        if line and '|' in line:
            table_lines.append(line)
    
    if len(table_lines) < 2:
        raise ValueError("텍스트 테이블을 찾을 수 없습니다.")
    
    # 헤더와 데이터 분리
    header_line = table_lines[0]
    data_lines = table_lines[2:]
    
    # 헤더 파싱
    headers = [col.strip() for col in header_line.split('|') if col]
    
    # 데이터 파싱
    data = []
    for line in data_lines:
        temp = re.sub(r'\\\|', '[[PIPE]]', line)
        
        parts = [p.strip() for p in temp.split('|') if p.strip()]
        row = [p.replace('[[PIPE]]', '|').replace('\\', '') for p in parts]

        if len(row) == len(headers):
            data.append(row)
        
    # 데이터프레임 생성
    df = pd.DataFrame(data, columns=headers)
    
    # count 컬럼이 있으면 숫자형으로 변환
    if 'count' in df.columns:
        df['count'] = pd.to_numeric(df['count'], errors='coerce')
    
    # 필요한 컬럼 확인
    required_cols = ['event_type', 'description', 'substitute'] #칼럼명 수정
    missing_cols = [col for col in required_cols if col not in df.columns]

    print(f"매핑 테이블 파싱 완료 - 컬럼: {list(df.columns)}")
    
    return df

def transform_mapping_dict(mapping_df) -> pd.DataFrame:
    """
    리스트 형태의 event_description을 가진 로그 데이터를 매핑 테이블 기반으로 변환하는 함수
    
    Args:
        mapping_df: 매핑 테이블 (event_type, description, result 컬럼 필요)

    Returns:
        pd.DataFrame: 변환된 데이터프레임
    """
    
    # 매핑 테이블을 딕셔너리로 변환 (빠른 룩업을 위해)
    mapping_dict = {}
    for _, row in mapping_df.iterrows():
        # (event_type, description) -> result 매핑
        key = (row['event_type'], row['description'])
        mapping_dict[key] = row['substitute']
    
    print(f"매핑 딕셔너리 생성 완료: {len(mapping_dict)} 개 규칙")
    
    return mapping_dict

def parse_description(desc):
    """event_description을 파싱하여 딕셔너리로 반환"""
    result = {}
    if pd.isna(desc) or desc == '':
        return result
    
    pairs = desc.split('|')
    for pair in pairs:
        if '=' in pair:
            key, value = pair.split('=', 1)
            result[key] = value
    return result

def transform_event(row):
    event_type = row['event_type']
    event_desc = row['event_desc_list']
    
    # A. event_type의 '_'를 '-'로 변환
    event_type_converted = event_type.replace('_', '-')
    
    # 1. 유지
    if event_type in maintain_list:
        parsed = parse_description(event_desc)
        
        detailed_info_list = []
        for value in parsed.values():
            converted_value = re.sub(r'(u\d+[A-Za-z0-9]*)(_|$)', r'\1-', value)
            detailed_info_list.append(converted_value)
            
        detailed_info = '_'.join(detailed_info_list)

        if detailed_info:
            return f"{event_type_converted}_{detailed_info}"
        else:
            return f"{event_type_converted}"
    
    # 2. 처리    
    if event_type == 'keypress':
        parsed = parse_description(event_desc)
        count = int(parsed.get('count', 0))
        count_str = '10+' if count > 10 else str(count)
        return f"keypress{count_str}"
    
    if event_type == 'cell_change':
        parsed = parse_description(event_desc)
        detailed_info = list(parsed.values())[0] if parsed else ''
        # content_spreadsheet_colad를 content-spreadsheet-colad로 변환
        converted_info = detailed_info.replace('content_spreadsheet_colad', 'content-spreadsheet-colad')
        return f"cell-change_{converted_info}"
    
    # 3. LLM
    if event_type in LLM_list:
        key = (event_type, event_desc)
        if key in LLM_mapping:
            return f"{event_type_converted}_{LLM_mapping[key]}"
        else:
            # 매핑이 없는 경우 원본 그대로
            return f"{event_type_converted}_{event_desc}"
    
    # 기타 경우
    if event_desc:
        return f"{event_type_converted}_{event_desc}"
    else:
        return f"{event_type_converted}"


In [ ]:
from itertools import product

left = range(1, 3)   # 1 ~ 2
right = range(1, 8)  # 1 ~ 7
pairs = product(left, right)

for pair in pairs:
    i, j = pair
    
    print(pair)
    problem_num = f'ps{i}_{j}'
    
    #데이터 전처리 경로
    path_2nd = f'input_data/2nd_data/us_{problem_num}.pkl'
    path_3rd = f'input_data/3rd_data/us_{problem_num}.pkl'

    #모델 학습 경로
    IRT_path = f'model_input/IRT/test_us_{problem_num}.csv'
    HW2V_path = f'model_input/HW2V/test_us_{problem_num}.txt'
    
    fin_data = pd.read_pickle(path_2nd)
    
    mapping_text_path = f'LLM/LLM_results/{problem_num}/step1_new_table.txt'
    mapping_df = parse_text_table(mapping_text_path)
    
    LLM_mapping = transform_mapping_dict(mapping_df)

    fin_data['processed_event'] = fin_data.apply(transform_event, axis=1)
    
    ## - 데이터 전처리 수정 - ##
    # 데이터에서 event_type이 'start'인 행 제거
    remove_list = ['start', 'button_next', 'button_ok', 'end']
    fin_data = fin_data[~fin_data['processed_event'].isin(remove_list)]
    ##############################################################
    
    # 람다 함수 사용
    fin_data['processed_event'] = fin_data['processed_event'].apply(
        lambda x: '_'.join(dict.fromkeys(x.split('_')))
    )
    
    result_df = fin_data[['SEQID', 'event_type', 'event_desc_list', 'timestamp', 'processed_event']]
    
    result_df.to_pickle(path_3rd)
    
    sequence_data = []

    for seqid, group in result_df.groupby('SEQID'):
        # timestamp 순으로 정렬된 processed_event들을 공백으로 연결
        events = group['processed_event'].tolist()
        seq_event = ' '.join(events)
        event_count = len(events)
        
        sequence_data.append({
            'SEQID': seqid,
            'seq_event': seq_event,
            'event_count': event_count
        })

    sequence_df = pd.DataFrame(sequence_data)
    
    sequence_df.to_csv(IRT_path, index=False)
    
    # action_sequence 열에서 빈 값이 아닌 데이터만 추출
    action_sequences = sequence_df['seq_event'].dropna()

    # 텍스트 파일로 저장 (각 행을 한 줄씩)
    with open(HW2V_path, 'w', encoding='utf-8') as f:
        for sequence in action_sequences:
            f.write(str(sequence) + '\n')

In [8]:
import pandas as pd
from itertools import product

left = range(1, 3)   # 1 ~ 2
right = range(1, 8)  # 1 ~ 7
pairs = product(left, right)

summary_list = []

for pair in pairs:
    i, j = pair
    
    print(pair)
    problem_num = f'ps{i}_{j}'
    
    data = pd.read_csv(f"/Users/jun_yeong/Desktop/Log_Process/01_Data/model_input/IRT/test_us_{problem_num}.csv")

    desc = data['event_count'].describe()
    desc['name'] = problem_num
    
    summary_list.append(desc)

combined_summary_list = pd.concat(summary_list, axis=1)
combined_summary_list.to_csv('/Users/jun_yeong/Desktop/Log_Process/01_Data/paper_stats/action_seq.csv')


(1, 1)
(1, 2)
(1, 3)
(1, 4)
(1, 5)
(1, 6)
(1, 7)
(2, 1)
(2, 2)
(2, 3)
(2, 4)
(2, 5)
(2, 6)
(2, 7)
